# 05. Pipeline RAG Completo (100% Local, Offline e Baseado em Evidências)

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Requisitos do Desafio:**
- Demonstração completa do fluxo: Pergunta -> Transformação -> Recuperação Semântica -> Contexto -> Síntese Fundamentada com Evidências.
- Respostas estritamente ancoradas nas avaliações reais dos clientes (Voice of Customer).
- Prevenção ativa de alucinações e tratamento de perguntas fora do escopo ou sem dados suficientes.
- **Arquitetura 100% Local e Gratuita:** sem consumo de APIs externas, chaves de nuvem ou custos de bilhetagem.

In [ ]:
import sys
import os
sys.path.append('../')

import chromadb
from chromadb.utils import embedding_functions
from src.rag_pipeline import PipelineRAG, SintetizadorLocalVoC, DEFAULT_PROMPT_TEMPLATE

DB_PATH = '../data/vector_db/'

# Instanciação do Pipeline RAG 100% Local
pipeline_rag = PipelineRAG(db_path=DB_PATH)
print(f"Pipeline RAG conectado com sucesso ao ChromaDB local!")
print(f"Total de documentos indexados na base: {pipeline_rag.collection.count():,}")

## 1. Engenharia de Prompt e Diretrizes de Fidelidade (Anti-Alucinação)
Visualizamos o template de prompt que orienta a síntese analítica e impõe a recusa caso as evidências não sustentem a resposta.

In [ ]:
print("=== TEMPLATE DE PROMPT E REGRAS DE GROUNDING ===")
print(DEFAULT_PROMPT_TEMPLATE)

## 2. Cenário 1: Investigação de Reclamações de Entrega (Edital)
Pergunta de negócio: *"O que os clientes relatam sobre problemas relacionados à entrega?"*  
Utilizamos o **Filtro de Metadados** (`review_score <= 2`) para recuperar estritamente experiências insatisfatórias.

In [ ]:
pergunta_1 = "O que os clientes relatam sobre problemas relacionados à entrega?"
resultado_1 = pipeline_rag.responder(
    pergunta=pergunta_1,
    n_results=4,
    where_filter={"review_score": {"$lte": 2}}
)

print(f"PERGUNTA: {resultado_1['pergunta']}\n")
print(resultado_1['resposta'])

## 3. Cenário 2: Busca por Experiências Positivas e Elogios (Edital)
Pergunta de negócio: *"Quais aspectos da experiência de compra são mais elogiados pelos clientes?"*  
Filtramos avaliações promotoras (`review_score >= 4`).

In [ ]:
pergunta_2 = "Quais aspectos da experiência de compra são mais elogiados pelos clientes?"
resultado_2 = pipeline_rag.responder(
    pergunta=pergunta_2,
    n_results=4,
    where_filter={"review_score": {"$gte": 4}}
)

print(f"PERGUNTA: {resultado_2['pergunta']}\n")
print(resultado_2['resposta'])

## 4. Cenário 3: Tratamento de Perguntas sem Evidências Suficientes / Fora de Escopo
Conforme o Item 4 do edital: *"Soluções baseadas em IA Generativa podem produzir respostas aparentemente convincentes mesmo quando não existem informações suficientes para sustentá-las. Sua solução deverá considerar situações em que a pergunta esteja fora do escopo da base de conhecimento."*  

Testamos uma pergunta esdrúxula: *"Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?"*.

In [ ]:
pergunta_fora_escopo = "Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?"
resultado_3 = pipeline_rag.responder(
    pergunta=pergunta_fora_escopo,
    n_results=3
)

print(f"PERGUNTA: {resultado_3['pergunta']}\n")
print(f"RESPOSTA DO SISTEMA:")
print(resultado_3['resposta'])

## 5. Rastreabilidade das Evidências (Auditoria Completa)
O sistema permite auditar detalhadamente todos os metadados, identificadores (`review_id`) e categorias retornadas pela base vetorial.

In [ ]:
print("Auditoria dos IDs recuperados para a Pergunta 1:")
for i, (doc_id, meta) in enumerate(zip(resultado_1['ids'], resultado_1['metadados']), 1):
    print(f"[{i}] ID do Review: {doc_id} | Nota no BD: {meta.get('review_score')}")